In [ ]:
import json
from datetime import datetime

from emu_renewal.inputs import DATA_PATH, END_VACC_THRESHOLD, get_country_vacc_data, get_filtered_indicator
from emu_renewal.run import find_run_end_time
from emu_renewal.selection import get_mob_countries, is_mostly_zeros, find_null_data, find_neg_data, has_repeats, has_outlier

In [ ]:
either_mob_avail = get_mob_countries()

In [ ]:
# Gather data
start_time = datetime(2020, 4, 1)
case_data = {}
death_data = {}
for c in either_mob_avail:
    vacc_data = get_country_vacc_data(c)
    end_time = find_run_end_time(vacc_data, END_VACC_THRESHOLD, c)
    case_data[c] = get_filtered_indicator("New_cases", start_time, end_time, c)
    death_data[c] = get_filtered_indicator("New_deaths", start_time, end_time, c)

In [ ]:
# Select countries based on data characteristics
n_repeats = 6
repeat_threshold = 1e-10
outlier_threshold = 2.0

no_cases = find_null_data(case_data)
no_deaths = find_null_data(death_data)
neg_cases = find_neg_data(case_data)
neg_deaths = find_neg_data(death_data)

case_repeats = [c for c, d in case_data.items() if has_repeats(d, n_repeats, repeat_threshold)]
death_repeats = [c for c, d in death_data.items() if has_repeats(d, n_repeats, repeat_threshold)]
case_outliers = [c for c, d in case_data.items() if has_outlier(d, outlier_threshold)]
death_outliers = [c for c, d in death_data.items() if has_outlier(d, outlier_threshold)]
mostly_zeroes = [c for c, d in case_data.items() if is_mostly_zeros(d)]

In [ ]:
excluded = list(set(no_cases + no_deaths + neg_cases + neg_deaths + case_repeats + death_repeats + case_outliers + death_outliers + mostly_zeroes))
included = [c for c in either_mob_avail if c not in excluded]
included.append("AUS")

In [ ]:
json.dump(included, open(DATA_PATH / "config/included.json", "w"))